# B3：SUMO CACC 编队协同巡航 —— TraCI 闭环

## 实验目标
- 理解 TraCI 实时控制接口
- 实现 CACC 编队跟驰算法
- 验证弦稳定性

## 通过判据
```python
assert max_spacing_error <= 1.0  # 编队 60s 无解体
```

## 确定性要求
- `traci.vehicle.setLaneChangeMode(veh_id, 0)` 关闭自主变道
- SUMO 启动加 `--seed 42`

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
print('✓ 导入成功')
# 检查 SUMO
import shutil
print('SUMO:', shutil.which('sumo'))

## 1. CACC 控制器
继承 `BaseController`，实现编队跟驰。

In [ ]:
class CACCController(BaseController):
    """协同自适应巡航：跟随前车，保持固定时距"""
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 0.8)
        self.kp, self.kd = 1.2, 0.8
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}
        lead = min(targets, key=lambda t: t['x'])
        gap_err = (lead['x'] - ego['x'] - 5.0) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}
print('✓ CACC 控制器定义完成')

## 2. TraCI 闭环仿真（框架）
> 完整 SUMO 路网需 `netgenerate` 生成，本节展示 TraCI 控制范式。

In [ ]:
# TraCI 控制范式（伪代码，完整路网下可运行）
traci_code = '''
import traci
# 启动 SUMO（确定性）
traci.start(['sumo', '-n', 'platoon.net.xml', '--seed', '42'])
# 关闭自主变道，由算法接管
for veh_id in traci.vehicle.getIDList():
    traci.vehicle.setLaneChangeMode(veh_id, 0)
# 闭环
controller = CACCController()
while traci.simulation.getMinExpectedNumber() > 0:
    traci.simulationStep()
    # 读状态 → 组 observation → controller.step() → traci.vehicle.setSpeed()
traci.close()
'''
print(traci_code)

## 3. 评测断言
```python
assert max_spacing_error <= 1.0
```

## 🏆 挑战任务
1. 用 `netgenerate` 生成直道，实际跑通 TraCI 闭环。
2. 对比不同 headway（0.6/0.8/1.0）对弦稳定性的影响。